# Exercise 1 — Merging Word Frequencies

This standalone notebook implements the two requested solutions using `defaultdict` and `Counter`.

It also adds:
- reusable functions with type hints and docstrings;
- input validation appropriate for frequency data;
- deterministic sorting by frequency descending, then word ascending for ties;
- examples for both the three-server and two-server cases;
- lightweight tests, including edge cases;
- a short complexity and implementation comparison.

## Problem

Each server returns a mapping of `word -> frequency`. We need to combine any number of these mappings by summing frequencies for identical words.

The final result should be sorted from highest to lowest frequency. For equal frequencies, this notebook sorts alphabetically so the result is deterministic.

In [1]:
from collections import Counter, defaultdict
from collections.abc import Mapping
from numbers import Integral

## Sample data

In [2]:
d1 = {'python': 10, 'java': 3, 'c#': 8, 'javascript': 15}
d2 = {'java': 10, 'c++': 10, 'c#': 4, 'go': 9, 'python': 6}
d3 = {'erlang': 5, 'haskell': 2, 'python': 1, 'pascal': 1}

## Shared helpers

The exercise itself can be solved with only a few lines, but validation makes these functions safer to reuse in real code.

Frequency values are required to be non-negative integers. `bool` is rejected explicitly because `bool` is technically a subclass of `int` in Python, but is not meaningful as a word frequency.

In [3]:
def _validate_sources(sources: tuple[Mapping[str, int], ...]) -> None:
    """Validate dictionaries/mappings used as word-frequency sources."""
    for source_index, source in enumerate(sources, start=1):
        if not isinstance(source, Mapping):
            raise TypeError(
                f"Source #{source_index} must be a mapping, "
                f"got {type(source).__name__}."
            )

        for word, count in source.items():
            if not isinstance(word, str):
                raise TypeError(
                    f"Word keys must be strings; got {word!r} "
                    f"({type(word).__name__}) in source #{source_index}."
                )

            if isinstance(count, bool) or not isinstance(count, Integral):
                raise TypeError(
                    f"Frequency for {word!r} must be an integer; "
                    f"got {count!r}."
                )

            if count < 0:
                raise ValueError(
                    f"Frequency for {word!r} cannot be negative; "
                    f"got {count}."
                )


def _sort_counts(counts: Mapping[str, int]) -> dict[str, int]:
    """Return a regular dict sorted by count descending, then word ascending."""
    return dict(sorted(counts.items(), key=lambda item: (-item[1], item[0])))

## Solution A — `defaultdict`

`defaultdict(int)` automatically supplies `0` for a word the first time it is encountered. That removes the need for `.get(word, 0)` or an explicit membership check.

In [4]:
def merge_with_defaultdict(
    *sources: Mapping[str, int],
    sort_result: bool = True,
) -> dict[str, int]:
    """Merge word-frequency mappings using ``defaultdict``.

    Parameters
    ----------
    *sources:
        Any number of mappings containing ``str -> non-negative int`` counts.
    sort_result:
        If True, return counts ordered by frequency descending and word
        ascending for ties. If False, preserve first-seen insertion order.

    Returns
    -------
    dict[str, int]
        Combined word frequencies.
    """
    _validate_sources(sources)

    merged: defaultdict[str, int] = defaultdict(int)

    for source in sources:
        for word, count in source.items():
            merged[word] += count

    return _sort_counts(merged) if sort_result else dict(merged)

In [5]:
merged_defaultdict = merge_with_defaultdict(d1, d2, d3)
merged_defaultdict

{'python': 17,
 'javascript': 15,
 'java': 13,
 'c#': 12,
 'c++': 10,
 'go': 9,
 'erlang': 5,
 'haskell': 2,
 'pascal': 1}

## Solution B — `Counter`

`Counter` is designed specifically for counting hashable objects. Its `update()` method adds incoming counts, making it a natural fit for this problem.

Using `update()` instead of repeatedly adding `Counter` objects also preserves keys whose count is zero.

In [6]:
def merge_with_counter(
    *sources: Mapping[str, int],
    sort_result: bool = True,
) -> dict[str, int]:
    """Merge word-frequency mappings using ``collections.Counter``.

    Parameters
    ----------
    *sources:
        Any number of mappings containing ``str -> non-negative int`` counts.
    sort_result:
        If True, return counts ordered by frequency descending and word
        ascending for ties. If False, preserve Counter iteration order.

    Returns
    -------
    dict[str, int]
        Combined word frequencies.
    """
    _validate_sources(sources)

    merged: Counter[str] = Counter()

    for source in sources:
        merged.update(source)

    return _sort_counts(merged) if sort_result else dict(merged)

In [7]:
merged_counter = merge_with_counter(d1, d2, d3)
merged_counter

{'python': 17,
 'javascript': 15,
 'java': 13,
 'c#': 12,
 'c++': 10,
 'go': 9,
 'erlang': 5,
 'haskell': 2,
 'pascal': 1}

## Expected results

In [8]:
expected_all = {
    'python': 17,
    'javascript': 15,
    'java': 13,
    'c#': 12,
    'c++': 10,
    'go': 9,
    'erlang': 5,
    'haskell': 2,
    'pascal': 1,
}

expected_first_two = {
    'python': 16,
    'javascript': 15,
    'java': 13,
    'c#': 12,
    'c++': 10,
    'go': 9,
}

print('All three servers:')
print(merge_with_counter(d1, d2, d3))

print('\nServers 1 and 2 only:')
print(merge_with_counter(d1, d2))

All three servers:
{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}

Servers 1 and 2 only:
{'python': 16, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9}


## Tests

These tests verify both required implementations against the examples and a few useful edge cases.

In [9]:
def run_tests() -> None:
    implementations = (merge_with_defaultdict, merge_with_counter)

    for merge in implementations:
        # Exercise examples
        assert merge(d1, d2, d3) == expected_all
        assert merge(d1, d2) == expected_first_two

        # Edge cases
        assert merge() == {}
        assert merge({}) == {}
        assert merge({'python': 0}) == {'python': 0}
        assert merge({'b': 2, 'a': 2}) == {'a': 2, 'b': 2}
        assert merge({'python': 2}, {'python': 3}) == {'python': 5}

        # Optional unsorted mode
        assert merge({'b': 1, 'a': 2}, sort_result=False) == {'b': 1, 'a': 2}

        # Invalid negative count
        try:
            merge({'python': -1})
        except ValueError:
            pass
        else:
            raise AssertionError('Negative frequencies should raise ValueError')

        # Invalid non-integer count
        try:
            merge({'python': 1.5})
        except TypeError:
            pass
        else:
            raise AssertionError('Non-integer frequencies should raise TypeError')

    # Both implementations should always agree for valid inputs.
    assert merge_with_defaultdict(d1, d2, d3) == merge_with_counter(d1, d2, d3)

    print('All tests passed.')


run_tests()

All tests passed.


## Complexity and recommendation

Let `N` be the total number of `(word, count)` pairs across all input mappings and `U` the number of unique words.

- Merging takes **O(N)** time and **O(U)** additional space.
- Sorting takes **O(U log U)** time.
- If `sort_result=False`, the overall merge remains **O(N)** time.

### Which solution should you prefer?

- **`defaultdict`** is excellent when you want explicit control over accumulation logic or may later compute something more complex than a simple sum.
- **`Counter`** is the most expressive choice here because the data represents counts directly. It communicates intent clearly and provides additional counting-oriented operations such as `most_common()`.

For this specific problem, the `Counter` implementation is the strongest default choice.